# TRELLIS.2 Inference on Google Colab
This notebook sets up the environment to run Microsoft's 4-billion parameter TRELLIS.2 model on a standard Google Colab (Tested using L4 GPU)

**Prerequisites:**
Before running this notebook, you must have a Hugging Face account and accept the usage agreements for the following gated models:
1. Background Removal: [briaai/RMBG-2.0](https://huggingface.co/briaai/RMBG-2.0)
2. Image Conditioning: [Facebook DINOv3](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m)

You will also need to add your Hugging Face Read Token to Colab's **Secrets** tab (the key icon on the left sidebar) and name it `HF_TOKEN`.

In [ ]:
# Clone the repository and install all custom CUDA extensions
!git clone -b main https://github.com/microsoft/TRELLIS.2.git --recursive

%cd /content/TRELLIS.2/

# Run the setup script to compile cumesh, o-voxel, and flexgemm
!. ./setup.sh --new-env --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm

# Downgrade libraries to avoid meta tensor and internal utility conflicts
!pip install "Pillow<10.0.0"
!pip install "transformers<5.0.0"

### 🛑 STOP AND RESTART RUNTIME
Because we installed specific versions of `Pillow` and `transformers`, Python will crash if you do not clear its active memory.

Go to the top menu: **Runtime > Restart session** (or Restart runtime).
Do **not** run the setup cell above again. Proceed directly to the cell below.

In [ ]:
# Re-enter the directory after the restart
%cd /content/TRELLIS.2/

from huggingface_hub import login
from google.colab import userdata

# Authenticate with Hugging Face to download the gated models
print("Logging into Hugging Face...")
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [7]:
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Can save GPU memory
import cv2
import imageio
from PIL import Image
import torch
from trellis2.pipelines import Trellis2ImageTo3DPipeline
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
import o_voxel

# 1. Setup Environment Map
envmap = EnvMap(torch.tensor(
    cv2.cvtColor(cv2.imread('assets/hdri/forest.exr', cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB),
    dtype=torch.float32, device='cuda'
))

# 2. Load Pipeline
pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()

# 3. Load Image & Run
image = Image.open("assets/example_image/T.png")
mesh = pipeline.run(image)[0]
mesh.simplify(16777216) # nvdiffrast limit

# 4. Render Video
video = render_utils.make_pbr_vis_frames(render_utils.render_video(mesh, envmap=envmap))
imageio.mimsave("sample.mp4", video, fps=15)

# 5. Export to GLB
glb = o_voxel.postprocess.to_glb(
    vertices            =   mesh.vertices,
    faces               =   mesh.faces,
    attr_volume         =   mesh.attrs,
    coords              =   mesh.coords,
    attr_layout         =   mesh.layout,
    voxel_size          =   mesh.voxel_size,
    aabb                =   [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target   =   1000000,
    texture_size        =   4096,
    remesh              =   True,
    remesh_band         =   1,
    remesh_project      =   0,
    verbose             =   True
)
glb.export("sample.glb", extension_webp=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py
/content/TRELLIS.2/trellis2/pipelines/trellis2_image_to_3d.py
(self, image: PIL.Image.Image, num_samples: int = 1, seed: int = 42, sparse_structure_sampler_params: dict = {}, shape_slat_sampler_params: dict = {}, tex_slat_sampler_params: dict = {}, preprocess_image: bool = True, return_latent: bool = False, pipeline_type: Optional[str] = None, max_num_tokens: int = 49152, sparse_structure_latent: Optional[torch.Tensor] = None, sparse_structure_latent_path: Optional[str] = None, use_sparse_structure_inversion: bool = False, sparse_structure_inversion_sampler_params: dict = {}) -> List[trellis2.representations.mesh.base.MeshWithVoxel]
1111111111111111


Rendering:   0%|          | 0/120 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/utils3d/torch/nerf.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:5015: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(
Rendering: 100%|██████████| 120/120 [01:51<00:00,  1.07it/s]


Original mesh: 2066339 vertices, 4161658 faces
After filling holes: 2066339 vertices, 4161658 faces
Building BVH for current mesh...Done
Cleaning mesh...


Building Sparse Grid: 100%|██████████| 6/6 [00:00<00:00, 61.63it/s]

Running Dual Contouring...


...

After remeshing: 4014491 vertices, 8030356 faces


Simplifying [thres=1.00e-07]: 100%|██████████| 7030356/7030356 [00:01<00:00, 4130439.36it/s]


After simplifying: 491946 vertices, 984620 faces
Done
Parameterizing new mesh...
Get 997 clusters after fast clustering


Gathering results from xatlas: 100%|██████████| 997/997 [00:00<00:00, 22774.40it/s]


Done
Sampling attributes...Done
Finalizing mesh...Done
Buffered data was truncated after reaching the output size limit.